# Trace Element Data Availability and Feature Selection

This notebook evaluates trace element data availability across the compiled lamproite dataset and identifies a common element suite suitable for subsequent multivariate analysis. Element retention is assessed against the corresponding retention of analyses and geographic coverage to balance geochemical dimensionality with sample size and spatial representation. The selected trace element suite and complete-case modelling dataset are exported for subsequent preprocessing and analysis.

## 1. Environment Setup

Import required libraries, define project directories and initialise the analysis environment.

In [ ]:
# =============================================================================
# Environment setup
# =============================================================================

import os
os.environ["OMP_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

import math
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

EXPORT_DIR = ROOT_DIR / "exports"
FIGURE_DIR = ROOT_DIR / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 99

print("Environment initialised.")
print(f"Root directory   : {ROOT_DIR}")
print(f"Raw data         : {RAW_DIR}")
print(f"Processed data   : {PROCESSED_DIR}")
print(f"Export directory : {EXPORT_DIR}")
print(f"Figure directory : {FIGURE_DIR}")

## 2. Import Dataset

Load the complete geochemical dataset, calculate elemental concentrations from oxide measurements where required, and define the trace element suite and withheld validation regions.

In [ ]:
# =============================================================================
# Import dataset
# =============================================================================

df = pd.read_csv(RAW_DIR / "Data_Input.csv", encoding="utf-8")

df["Ti"] = df["TiO2"] * 0.5994 * 1e4
df["K"] = df["K2O"] * 0.8301 * 1e4
df["P"] = df["P2O5"] * 0.4364 * 1e4

TRACE_ELEMENTS = [
    "Cs","Rb","Ba","Th","U","Nb","Ta","K","La","Ce","Pb","Pr","Sr","Nd","P",
    "Sm","Zr","Hf","Eu","Ti","Gd","Tb","Dy","Y","Ho","Er","Tm","Yb","Lu"
]

VALIDATION_COUNTRY = "China"

print("Dataset successfully loaded.")
print(f"Analyses: {len(df)}")

## 3. Dataset Inspection

Inspect the imported dataset to verify its structure, identify missing values and confirm that all required metadata and geochemical variables are present prior to analysis.



In [ ]:
# =============================================================================
# Trace-element completeness by country and tectonic series
# =============================================================================

df["Dataset"] = np.where(
    df["Country"].eq(VALIDATION_COUNTRY),
    "Validation",
    "Development"
)

series_coverage = (
    df.groupby(["Dataset", "Series", "Country"], dropna=False)
    .apply(lambda d: pd.Series({
        "Analyses": int(len(d)),
        "Completeness (%)": 100 * d[TRACE_ELEMENTS].notna().sum().sum()
                            / d[TRACE_ELEMENTS].size
    }), include_groups=False)
    .reset_index()
    .sort_values(
        ["Dataset", "Series", "Completeness (%)", "Analyses"],
        ascending=[True, True, False, False]
    )
)

series_coverage["Analyses"] = series_coverage["Analyses"].astype(int)

display(series_coverage)
dev_df = df[df["Dataset"].eq("Development")].copy()

## 4. Trace Element Data Quality Assessment

Assess the completeness and validity of the trace element dataset prior to data cleaning. The inspection summarises missing values and identifies any non-positive concentrations that would prevent subsequent log transformation.

In [ ]:
# =============================================================================
# Histogram of missing trace elements per analysis
# =============================================================================

missing_per_row = dev_df[TRACE_ELEMENTS].isna().sum(axis=1)

fig, ax = plt.subplots(figsize=(8,5))
ax.hist( missing_per_row, bins=np.arange(-0.5, len(TRACE_ELEMENTS)+1.5, 1),
    edgecolor="black", color="lightgrey" )
ax.set_xlabel("Number of missing trace elements", fontsize=12)
ax.set_ylabel("Number of analyses", fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
outpath = FIGURE_DIR / "04_Missing_trace_elements_histogram.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =============================================================================
# Histogram of missing analyses per trace element
# =============================================================================

missing_per_element = dev_df[TRACE_ELEMENTS].isna().sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10,5))
ax.bar( missing_per_element.index, missing_per_element.values, 
        edgecolor="black", color="lightgrey" )
ax.set_xlabel("Trace element", fontsize=12)
ax.set_ylabel("Number of analyses with missing values", fontsize=12)
ax.tick_params(axis="x", rotation=90)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
outpath = FIGURE_DIR / "04_Missing_analysis_per_element_histogram.png"
plt.show()

In [ ]:
# =============================================================================
# Table 01: Trace element data coverage
# =============================================================================

trace_element_coverage = pd.DataFrame({
    "Element": TRACE_ELEMENTS,
    "Available analyses": dev_df[TRACE_ELEMENTS].notna().sum().values,
    "Missing analyses": dev_df[TRACE_ELEMENTS].isna().sum().values,
    "Completeness (%)": (dev_df[TRACE_ELEMENTS].notna().mean() * 100).values
})

trace_element_coverage = (
    trace_element_coverage
    .sort_values("Completeness (%)", ascending=False)
    .reset_index(drop=True)
)

display(trace_element_coverage)

outpath = EXPORT_DIR / "01_trace_element_coverage.csv"
trace_element_coverage.to_csv(outpath, index=False)

print(f"Saved: {outpath}")

In [ ]:
# =============================================================================
# Table 02: Dataset retention across trace element thresholds
# =============================================================================

element_missing_order = ( dev_df[TRACE_ELEMENTS].isna().sum().sort_values(ascending=True) )
element_thres_results = []
for n_elements in range(len(TRACE_ELEMENTS), 0, -1):
    retained_elements = element_missing_order.index[:n_elements].tolist()
    temp_df = dev_df.dropna(subset=retained_elements).copy()
    series_counts = temp_df["Series"].value_counts()
    n_countries = temp_df["Country"].nunique(dropna=True)

    element_thres_results.append({
        "Retained_elements": n_elements, "Elements": ", ".join(retained_elements),
        "Analyses": len(temp_df), "Orogenic": series_counts.get("Orogenic", 0),
        "Anorogenic": series_counts.get("Anorogenic", 0), "Countries": n_countries,
        "Analyses_x_elements": len(temp_df) * n_elements })

element_thres_df = pd.DataFrame(element_thres_results)
element_thres_df["ΔAnalyses"] = element_thres_df["Analyses"].diff()
element_thres_df["ΔCountries"] = element_thres_df["Countries"].diff()
element_thres_df["ΔAnalyses_x_elements"] = ( element_thres_df["Analyses_x_elements"].diff()
)
display(element_thres_df)

outpath = EXPORT_DIR / "02_element_retention_summary.csv"
element_thres_df.to_csv(outpath, index=False)

print(f"Saved: {outpath}")

In [ ]:
# =============================================================================
# Figure 05: Trace element retention and marginal gain
# =============================================================================

gain_df = element_thres_df.copy()
gain_df["ΔAnalyses_norm"] = ( gain_df["ΔAnalyses"] / gain_df["ΔAnalyses"].max() )
gain_df["ΔCountries_norm"] = ( gain_df["ΔCountries"] / gain_df["ΔCountries"].max() )
gain_df["ΔAnalyses_x_elements_norm"] = ( gain_df["ΔAnalyses_x_elements"] 
                                         / gain_df["ΔAnalyses_x_elements"].abs().max() )
retn_df = element_thres_df.copy()
retn_df["Analyses_retained"] = ( retn_df["Analyses"] / retn_df["Analyses"].max() )
retn_df["Countries_retained"] = ( retn_df["Countries"] / retn_df["Countries"].max() )
retn_df["Analyses_x_elements_retained"] = ( retn_df["Analyses_x_elements"]
                / retn_df["Analyses_x_elements"].max() )

fig, axes = plt.subplots( nrows=2, ncols=1, figsize=(10, 9), sharex=True)
ax1, ax2 = axes

ax1.plot( gain_df["Retained_elements"], gain_df["ΔAnalyses_norm"],
          label=r"$\Delta$ Analyses", marker="o" )
ax1.plot( gain_df["Retained_elements"], gain_df["ΔCountries_norm"],
          label=r"$\Delta$ Countries",  marker="o" )
ax1.plot( gain_df["Retained_elements"], gain_df["ΔAnalyses_x_elements_norm"],
          label=r"$\Delta$ Analyses $\times$ elements", marker="o" )
ax1.axhline(0, color="grey", linewidth=0.8)
ax1.set_ylabel("Normalised marginal change", fontsize=12)
ax1.grid(alpha=0.3)
ax1.legend()

ax2.plot( retn_df["Retained_elements"], retn_df["Analyses_retained"], label="Analyses", marker="o")
ax2.plot( retn_df["Retained_elements"], retn_df["Countries_retained"], label="Countries", marker="o")
ax2.plot( retn_df["Retained_elements"], retn_df["Analyses_x_elements_retained"], label="Analyses × elements", marker="o")
ax2.set_xlabel("Number of retained trace elements", fontsize=12)
ax2.set_ylabel("Fraction of maximum retained", fontsize=12)
ax2.set_xticks(retn_df["Retained_elements"])
ax2.set_ylim(0, 1.05)
ax2.invert_xaxis()
ax2.grid(alpha=0.3)
ax2.legend()
plt.tight_layout()
outpath = FIGURE_DIR / "05_trace_element_retention.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {outpath}")

In [ ]:
# =============================================================================
# Final trace element selection and export
# =============================================================================

N_RETAINED_ELEMENTS = 14

retained_elements = (
    element_missing_order
    .index[:N_RETAINED_ELEMENTS]
    .tolist()
)

retained_df = dev_df.dropna(subset=retained_elements).copy()

retained_geography = (
    retained_df
    .groupby(["Series", "Country"], dropna=False)
    .size()
    .reset_index(name="Analyses")
    .sort_values(["Series", "Country"])
)

print(f"Number of retained elements : {len(retained_elements)}")
print(f"Number of retained analyses : {len(retained_df)}")

print("\nRetained elements:")
print(retained_elements)

print("\nRetained Series × Country combinations:")
display(retained_geography)

geo_outpath = EXPORT_DIR / "03_retained_geographic_coverage.csv"
element_outpath = EXPORT_DIR / "04_retained_trace_elements.csv"
model_outpath = PROCESSED_DIR / "Modelling_Dataset.csv"

retained_geography.to_csv(geo_outpath, index=False)

pd.DataFrame({
    "Element": retained_elements
}).to_csv(element_outpath, index=False)

retained_df.to_csv(model_outpath, index=False)

print(f"Saved: {geo_outpath}")
print(f"Saved: {element_outpath}")
print(f"Saved: {model_outpath}")